# Pipeline de preprocesamiento y entrenamiento, paso a paso

Este notebook **no reimplementa nada**: cada celda llama a las funciones de `app/ml_engine/src` (los mismos módulos que usan el entrenamiento y el servicio), y solo agrega gráficas con matplotlib para ver qué hace cada bloque del pipeline (`docs/ml-engine/preprocessing-pipeline.md`, B1-B10).

- Se corre en el contenedor `notebook` (`docker compose --profile notebook up notebook`), desde `app/ml_engine`.
- Las features se leen de la caché `artifacts/feature-cache` (la crea la primera corrida; si no existe, la sección 9 tarda unos 8 minutos).
- La parte de entrenamiento (secciones 11-14) la corres tú. **No se exporta ningún modelo**: no se llama a `write_artifact` ni a `write_report`.
- La sección 14 evalúa Test. Por diseño (G7), Test se mira **una sola vez** por versión del modelo; para comparar parámetros usa la validación cruzada (secciones 11-13).

## 0. Configuración

In [ ]:
import os
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

from src.classical.extractor import extract_radiomics
from src.deep_learning.bundle import sha256_of
from src.deep_learning.extractor import DeepFeatureExtractor
from src.errors import MLEngineError
from src.preprocessing.brain_mask import bounding_box, foreground_mask, region_of_interest
from src.preprocessing.geometry import crop_square, resize_square, square_box
from src.preprocessing.loading import load_mri_image
from src.preprocessing.prepare import prepare_image
from src.preprocessing.quality import check_quality, measure_quality
from src.training import run
from src.training.ablation import ablate_blocks
from src.training.augmentation import augment, copy_rng, draw, keeps_augmentation
from src.training.calibration import calibrate
from src.training.config import TrainingConfig, apply_round, resolved_config
from src.training.dataset import load_samples, split_samples
from src.training.evaluation import evaluate
from src.training.features import copy_rows
from src.training.folds import assign_folds, fold_record, fold_splits, used_per_fold_and_class
from src.training.guarantees import leakage_guarantees
from src.training.model import (
    build_model,
    candidate_classifiers,
    compare_classifiers,
    cross_validate_predictions,
)
from src.training.out_of_fold import out_of_fold_metrics
from src.training.shortcut import metadata_baseline, metadata_matrix, recall_by_source_size

plt.rcParams["figure.dpi"] = 110
LABELS = run.LABELS
print("raíz:", ROOT)

Parámetros de la corrida. Cambia `ROUND_FILE` para probar otra ronda (`rounds/<versión>.toml`); deja `None` para los valores por defecto. `CACHE_DIR` apunta a la misma caché que usa el `trainer`, así no se recalculan las features.

In [ ]:
DATA_DIR = Path("src/data")
MANIFEST = DATA_DIR / "processed/dedup_20261028T234450903626Z/manifest.csv"
WEIGHTS = Path("weights")
CACHE_DIR = Path("artifacts/feature-cache")
ROUND_FILE = Path("rounds/0.1.0.toml")

config = TrainingConfig(
    data_dir=DATA_DIR,
    manifest=MANIFEST,
    backbone_bundle=WEIGHTS,
    out_dir=Path("artifacts/notebook"),
    cache_dir=CACHE_DIR,
)
if ROUND_FILE is not None:
    config = apply_round(config, ROUND_FILE)

resolved = resolved_config(config)
for section in ("run", "preprocessing", "augmentation", "model"):
    print(section, resolved[section])

## 1. Dataset y split sin fugas (B1, B2)

`load_samples` lee las imágenes conservadas por la deduplicación (B1, `manifest.csv`). `split_samples` deja Test intacto y quita de Train toda imagen cuyo `group_id` (grupo de casi-duplicados) también esté en Test (B2, garantía G1).

In [ ]:
samples = load_samples(config.data_dir, config.manifest)
split = split_samples(samples)

train_counts = Counter(s.label.value for s in split.train)
test_counts = Counter(s.label.value for s in split.test)
removed = {label.value: n for label, n in split.removed_per_class.items()}
print("imágenes conservadas:", len(samples), "| Train:", len(split.train), "| Test:", len(split.test))
print("quitadas de Train por compartir grupo con Test:", removed)

x = np.arange(len(LABELS))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(x - 0.27, [train_counts[l] for l in LABELS], 0.27, label="Train (final)")
ax.bar(x, [removed.get(l, 0) for l in LABELS], 0.27, label="quitadas por B2")
ax.bar(x + 0.27, [test_counts[l] for l in LABELS], 0.27, label="Test")
ax.set_xticks(x, LABELS)
ax.set_ylabel("imágenes")
ax.set_title("Imágenes por clase")
ax.legend()
plt.show()

shared = {s.group_id for s in split.train} & {s.group_id for s in split.test}
print("G1, group_id en Train y Test a la vez:", len(shared))

## 2. Geometría de origen por clase

El tamaño de archivo y de imagen depende de la fuente de cada clase (sección 2 del documento del pipeline): casi todas las imágenes con tumor son 512x512 y casi ninguna `no_tumor` lo es. Si el modelo aprendiera esto, estaría usando un atajo. B6 (recorte cuadrado) y B10 (auditoría de atajos) existen por esto.

In [ ]:
meta = metadata_matrix(split.train)
labels_train = np.array([s.label.value for s in split.train])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label in LABELS:
    rows = labels_train == label
    axes[0].scatter(meta[rows, 0], meta[rows, 1], s=8, alpha=0.5, label=label)
    axes[1].hist(meta[rows, 2] / 1024, bins=40, alpha=0.5, label=label)
axes[0].set_xlabel("ancho (px)")
axes[0].set_ylabel("alto (px)")
axes[0].set_title("Tamaño de origen (Train)")
axes[1].set_xlabel("tamaño de archivo (KiB)")
axes[1].set_title("Peso del archivo (Train)")
axes[0].legend()
plt.tight_layout()
plt.show()

for label in LABELS:
    rows = labels_train == label
    square512 = np.all(meta[rows, :2] == (512, 512), axis=1).mean()
    print(f"{label:11s} 512x512: {square512:.1%}")

## 3. Una imagen, bloque por bloque (B3-B6)

Elige una imagen con `CLASS` y `INDEX`. Cada paso llama a la función del módulo correspondiente; al final se comprueba que el resultado es idéntico a `prepare_image`, que es lo que corren el entrenamiento y el servicio.

In [ ]:
CLASS = "no_tumor"
INDEX = 0

sample = [s for s in split.train if s.label.value == CLASS][INDEX]
print(sample.path)

gray = load_mri_image(sample.path)  # B3: carga canónica, float32 en [0, 1]
brain = foreground_mask(gray)  # B5: mitad del umbral de Otsu, componente mayor, huecos rellenos
quality = measure_quality(gray, mask=brain)  # B4: SNR y entropía dentro de la máscara
mask = region_of_interest(brain, config.preprocessing.min_mask_fraction)
box = square_box(mask, config.preprocessing.crop_margin)  # B6: cuadrado centrado + margen
square = crop_square(gray, box)
cnn_image = resize_square(square, config.preprocessing.cnn_size)

print(f"forma original {gray.shape}, rango [{gray.min():.3f}, {gray.max():.3f}]")
print(f"máscara cubre {brain.mean():.1%} de la imagen")
print(f"SNR {quality.snr:.2f} (mín {config.preprocessing.thresholds.min_snr}), "
      f"entropía {quality.entropy:.2f} (mín {config.preprocessing.thresholds.min_entropy})")

In [ ]:
rows, cols = bounding_box(mask)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title(f"B3 carga\n{gray.shape[1]}x{gray.shape[0]}")
axes[1].imshow(gray, cmap="gray")
axes[1].imshow(np.ma.masked_where(~brain, brain), cmap="autumn", alpha=0.35)
axes[1].set_title("B5 máscara cerebral")
axes[2].imshow(gray, cmap="gray")
axes[2].add_patch(Rectangle((cols.start, rows.start), cols.stop - cols.start, rows.stop - rows.start,
                            fill=False, edgecolor="cyan", linewidth=1.5))
axes[2].add_patch(Rectangle((box.left, box.top), box.side, box.side,
                            fill=False, edgecolor="yellow", linewidth=1.5, linestyle="--"))
axes[2].set_title("B6 caja (cian) y cuadrado (amarillo)")
axes[3].imshow(cnn_image, cmap="gray", vmin=0, vmax=1)
axes[3].set_title(f"B6 entrada CNN\n{cnn_image.shape[1]}x{cnn_image.shape[0]}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

prepared = prepare_image(gray, config.preprocessing)
print("igual a prepare_image:", np.array_equal(prepared.cnn_image, cnn_image), np.array_equal(prepared.mask, mask))

## 4. Ejemplos por clase

Original y entrada CNN (B6) de varias imágenes por clase. Fíjate en que la entrada CNN conserva la proporción del cerebro aunque la imagen original no sea cuadrada.

In [ ]:
PER_CLASS = 4
fig, axes = plt.subplots(len(LABELS), 2 * PER_CLASS, figsize=(2 * PER_CLASS * 1.6, len(LABELS) * 1.8))
for row, label in enumerate(LABELS):
    picks = [s for s in split.train if s.label.value == label][:PER_CLASS]
    for i, s in enumerate(picks):
        image = load_mri_image(s.path)
        axes[row, 2 * i].imshow(image, cmap="gray")
        axes[row, 2 * i].set_title(f"{image.shape[1]}x{image.shape[0]}", fontsize=7)
        axes[row, 2 * i + 1].imshow(prepare_image(image, config.preprocessing).cnn_image, cmap="gray", vmin=0, vmax=1)
        axes[row, 2 * i + 1].set_title("CNN", fontsize=7)
    axes[row, 0].set_ylabel(label)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 5. Puerta de calidad sobre todo Train (B4)

Mide SNR y entropía en cada imagen de Train y las compara con los umbrales calibrados (D17). Los umbrales se calibraron solo con Train (G2), así que aquí no se usa Test. Tarda alrededor de un minuto.

In [ ]:
snr, entropy, rejected = [], [], []
for s in split.train:
    image = load_mri_image(s.path)
    q = measure_quality(image)
    snr.append(q.snr)
    entropy.append(q.entropy)
    try:
        check_quality(q, config.preprocessing.thresholds)
    except MLEngineError as error:
        rejected.append((s.path.name, str(error)))
snr, entropy = np.array(snr), np.array(entropy)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label in LABELS:
    rows = labels_train == label
    axes[0].hist(snr[rows], bins=40, alpha=0.5, label=label)
    axes[1].hist(entropy[rows], bins=40, alpha=0.5, label=label)
axes[0].axvline(config.preprocessing.thresholds.min_snr, color="red", linestyle="--", label="mínimo")
axes[1].axvline(config.preprocessing.thresholds.min_entropy, color="red", linestyle="--", label="mínimo")
axes[0].set_title("SNR (Train)")
axes[1].set_title("Entropía en bits (Train)")
axes[0].legend()
plt.tight_layout()
plt.show()
print("rechazadas por la puerta de calidad:", len(rejected), rejected[:5])

## 6. Aumento de datos solo en Train (B7)

Cada copia se sortea con la semilla de la corrida, el hash del archivo y el índice de copia (`copy_rng`), así que no depende del orden. Solo cambia la entrada CNN; las radiomics de una copia son las del original. Las copias nunca son filas de validación ni de Test (G3).

In [ ]:
N_COPIES = 6
digest = sha256_of(sample.path)
params = [draw(config.augmentation, copy_rng(config.seed, digest, i)) for i in range(N_COPIES)]

fig, axes = plt.subplots(1, N_COPIES + 1, figsize=(2.2 * (N_COPIES + 1), 2.6))
axes[0].imshow(cnn_image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("original", fontsize=8)
for ax, p in zip(axes[1:], params):
    ax.imshow(augment(cnn_image, p), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"rot {p.rotation_deg:+.1f}° esc {p.scale:.2f}\nflip {p.flip} b {p.brightness:.2f} c {p.contrast:.2f}", fontsize=7)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Tensor CNN y embedding (B8)

La imagen gris se replica en 3 canales y se normaliza con la media y desviación del backbone, leídas del bundle (D20). El backbone congelado devuelve un vector de 1280 valores.

In [ ]:
extractor = DeepFeatureExtractor.from_bundle(config.backbone_bundle, config.preprocessing)
manifest = extractor.manifest
print(f"backbone {manifest.backbone}, entrada {manifest.input_size}, media {manifest.mean}, std {manifest.std}")

tensor = extractor.to_tensor(cnn_image)
embedding = extractor.extract(cnn_image)
print("tensor", tensor.shape, "| embedding", embedding.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for channel in range(3):
    axes[0].hist(tensor[channel].ravel(), bins=60, alpha=0.4, label=f"canal {channel}")
axes[0].set_title("Valores del tensor normalizado")
axes[0].legend()
axes[1].plot(embedding, linewidth=0.6)
axes[1].set_title("Embedding CNN (1280)")
plt.tight_layout()
plt.show()

## 8. Features radiómicas

GLCM, LBP, wavelet e intensidad dentro de la máscara, sobre la imagen a su resolución original (no sobre la entrada CNN).

In [ ]:
radiomic = extract_radiomics(prepared.radiomics_image, prepared.mask, config.radiomics)
families = [name.split("_")[0] for name in radiomic.names]
colors = {f: plt.cm.tab10(i) for i, f in enumerate(dict.fromkeys(families))}

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar(range(len(radiomic.values)), radiomic.values, color=[colors[f] for f in families])
ax.set_xticks(range(len(radiomic.names)), radiomic.names, rotation=90, fontsize=6)
ax.set_yscale("symlog")
ax.set_title(f"{len(radiomic.values)} features radiómicas (color por familia)")
plt.tight_layout()
plt.show()

## 9. Features de todo el dataset

`run._train_features` y `run._test_features` son los mismos pasos que usa el entrenamiento (Test solo en una corrida `--final`): corren B3-B8 y las radiomics para cada imagen de Train y Test, y las copias aumentadas de Train, leyendo de la caché cuando la configuración no cambió.

In [ ]:
extraction = run._extraction(config)
features = run._train_features(config, extraction, split)
test_table, test_skipped = run._test_features(config, extraction, split)
train_table = features.table
x_train = train_table.matrix
y_train = np.array([s.label.value for s in train_table.samples])
n_radiomic = train_table.radiomics.shape[1]
print("Train", x_train.shape, "| Test", test_table.matrix.shape, "| copias", features.copy_cnn.shape)
print("descartadas:", {"Train": len(features.skipped), "Test": len(test_skipped)})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
blocks = {"Radiomics (45)": train_table.radiomics, "CNN (1280)": train_table.cnn}
for ax, (name, block) in zip(axes, blocks.items()):
    points = PCA(n_components=2, random_state=config.seed).fit_transform(StandardScaler().fit_transform(block))
    for label in LABELS:
        rows = y_train == label
        ax.scatter(points[rows, 0], points[rows, 1], s=6, alpha=0.5, label=label)
    ax.set_title(f"PCA 2D de {name}, solo para visualizar")
axes[0].legend()
plt.tight_layout()
plt.show()

## 10. Folds agrupados (B9)

Los folds se asignan una vez, antes de las features, con `StratifiedGroupKFold` sobre `group_id`: un grupo de casi-duplicados nunca queda a ambos lados de un split.

In [ ]:
fold_of = assign_folds(split.train, config.n_splits, config.seed)
fold_of_row = np.array([fold_of[s.path] for s in train_table.samples])
splits = fold_splits(fold_of_row, config.n_splits)
folds = fold_record(split.train, fold_of, {s.path for s in train_table.samples}, config.data_dir)
counts = used_per_fold_and_class(folds, config.n_splits, LABELS)

matrix = np.array([[counts[f][l] for l in LABELS] for f in sorted(counts)])
fig, ax = plt.subplots(figsize=(6, 3))
image = ax.imshow(matrix, cmap="Blues")
for (i, j), value in np.ndenumerate(matrix):
    ax.text(j, i, value, ha="center", va="center")
ax.set_xticks(range(len(LABELS)), LABELS)
ax.set_yticks(range(len(counts)), [f"fold {f}" for f in sorted(counts)])
ax.set_title("Imágenes usadas por fold y clase")
plt.show()

group_folds = {}
for s in split.train:
    group_folds.setdefault(s.group_id, set()).add(fold_of[s.path])
print("grupos repartidos en más de un fold:", sum(len(v) > 1 for v in group_folds.values()))

## 11. Selección del clasificador (entrenamiento, lo corres tú)

Compara regresión logística, SVM RBF y random forest con validación cruzada agrupada y elige por macro-F1 medio. Todo lo que se ajusta (scaler, PCA, clasificador) vive dentro del `Pipeline`, así que se reajusta en cada fold (G8).

In [ ]:
selection = compare_classifiers(x_train, y_train, n_radiomic, splits, config.seed, config.model)
names = list(selection.scores)
means = [selection.scores[n].mean_macro_f1 for n in names]
stds = [selection.scores[n].std_macro_f1 for n in names]

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(names, means, yerr=stds, capsize=5, color=["tab:green" if n == selection.best else "tab:gray" for n in names])
ax.set_ylim(min(means) - 0.1, 1)
ax.set_ylabel("macro-F1 (CV)")
ax.set_title(f"Selección: {selection.best}")
plt.show()
for n in names:
    s = selection.scores[n]
    print(f"{n:20s} macro-F1 {s.mean_macro_f1:.3f} ± {s.std_macro_f1:.3f} | por fold {np.round(s.fold_macro_f1, 3)}")

## 12. Aumento de datos y ablación (B7, B9)

Se repite la validación del clasificador elegido con las copias aumentadas como filas de ajuste. El aumento se queda solo si mejora el macro-F1 al menos una desviación estándar (D21). Después se mide cuánto aporta cada bloque de features.

In [ ]:
classifier = candidate_classifiers(config.seed, config.model.classifier_params)[selection.best]
plain = run._Rows(x_train, y_train, splits)
augmented_rows = run._with_copies(
    plain,
    fold_of_row,
    copy_rows(train_table, features.copy_cnn, config.augmentation.copies),
    config.augmentation.copies,
    config.n_splits,
)
without = selection.scores[selection.best]
with_copies, copies_predictions = cross_validate_predictions(
    classifier, augmented_rows.x, augmented_rows.y, n_radiomic, augmented_rows.splits,
    len(y_train), config.seed, config.model.pca_components,
)
augmented = keeps_augmentation(without.mean_macro_f1, without.std_macro_f1, with_copies.mean_macro_f1)
rows, chosen = (augmented_rows, with_copies) if augmented else (plain, without)
print(f"sin copias {without.mean_macro_f1:.3f} ± {without.std_macro_f1:.3f} | "
      f"con copias {with_copies.mean_macro_f1:.3f} ± {with_copies.std_macro_f1:.3f} -> aumento {'ON' if augmented else 'OFF'}")

ablation = ablate_blocks(rows.x, rows.y, n_radiomic, classifier, rows.splits,
                         chosen.mean_macro_f1, config.seed, config.model.pca_components)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(ablation), list(ablation.values()), color=["tab:blue", "tab:orange", "tab:green"])
ax.set_ylim(min(ablation.values()) - 0.1, 1)
ax.set_ylabel("macro-F1 (CV)")
ax.set_title("Ablación de bloques")
plt.show()

## 13. Métricas por clase en validación cruzada y auditoría de atajos

Predicciones out-of-fold del clasificador elegido, sin calibrar: cada imagen de Train la predice el modelo del fold que no la vio. **Esta es la sección para comparar parámetros.** Luego, el baseline que solo ve ancho, alto y peso del archivo (B10): si llega a 0.70 de balanced accuracy, la clase se puede adivinar sin anatomía.

In [ ]:
predictions = copies_predictions if augmented else selection.predictions[selection.best]
oof = out_of_fold_metrics(y_train, predictions, LABELS)
print(f"out-of-fold: macro-F1 {oof.macro_f1:.3f}, balanced accuracy {oof.balanced_accuracy:.3f}")
for label, m in oof.per_class.items():
    print(f"  {label:11s} precision {m.precision:.3f} recall {m.recall:.3f} f1 {m.f1:.3f} n={m.support}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(np.array(oof.confusion_matrix), display_labels=LABELS).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Validación cruzada (Train), filas = verdad")
plt.xticks(rotation=30)
plt.show()

baseline = metadata_baseline(train_table.samples, y_train, splits, config.seed)
print(f"baseline solo metadatos: {baseline['mean_balanced_accuracy']:.3f} ± {baseline['std_balanced_accuracy']:.3f}"
      f" (aviso a partir de {baseline['warning_level']})")
for g in leakage_guarantees(split, rows.splits, len(y_train)):
    print(g["id"], g["status"], "-", g["guarantee"])

## 14. Calibración y evaluación en Test (una sola vez)

Calibra el modelo (sigmoide, un pipeline por fold) y evalúa Test **una vez**. No se guarda ningún artefacto. Si después de mirar Test cambias parámetros, el número de Test deja de ser una medida limpia: compara rondas con la sección 13 y deja Test para la versión final.

In [ ]:
model = build_model(classifier, n_radiomic, train_table.cnn.shape[1], config.seed, config.model.pca_components)
calibrated = calibrate(model, rows.x, rows.y, rows.splits)

y_test = np.array([s.label.value for s in test_table.samples])
test_predictions = calibrated.predict(test_table.matrix)
test = evaluate(y_test, test_predictions, LABELS)
print(f"Test: macro-F1 {test.macro_f1:.3f}, accuracy {test.accuracy:.3f}, balanced accuracy {test.balanced_accuracy:.3f}")
for label, m in test.per_class.items():
    print(f"  {label:11s} precision {m.precision:.3f} recall {m.recall:.3f} f1 {m.f1:.3f} n={m.support}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(np.array(oof.confusion_matrix), display_labels=LABELS).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("CV out-of-fold (Train)")
ConfusionMatrixDisplay(np.array(test.confusion_matrix), display_labels=LABELS).plot(ax=axes[1], cmap="Oranges", colorbar=False)
axes[1].set_title("Test")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print("recall de no_tumor en Test por tamaño de origen:",
      recall_by_source_size(test_table.samples, y_test, test_predictions, run.NO_TUMOR))

In [ ]:
probabilities = calibrated.predict_proba(test_table.matrix)
confidence = probabilities.max(axis=1)
correct = test_predictions == y_test

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist([confidence[correct], confidence[~correct]], bins=20, stacked=True, label=["acierto", "error"])
axes[0].set_xlabel("confianza (probabilidad máxima)")
axes[0].set_title("Confianza en Test")
axes[0].legend()
bins = np.linspace(0, 1, 11)
which = np.clip(np.digitize(confidence, bins) - 1, 0, 9)
accuracy_per_bin = [correct[which == b].mean() if np.any(which == b) else np.nan for b in range(10)]
axes[1].plot(bins[:-1] + 0.05, accuracy_per_bin, marker="o", label="modelo")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfecto")
axes[1].set_xlabel("confianza")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Calibración en Test")
axes[1].legend()
plt.tight_layout()
plt.show()

## 15. Errores en Test: tumores clasificados como sanos

Muestra imágenes de Test con tumor que el modelo predijo `no_tumor`, el error clínicamente más grave (D24). La entrada CNN es lo que realmente ve el modelo.

In [ ]:
MAX_SHOWN = 8
missed = [i for i, (t, p) in enumerate(zip(y_test, test_predictions)) if t != run.NO_TUMOR and p == run.NO_TUMOR]
print("tumores predichos como no_tumor:", len(missed))

shown = missed[:MAX_SHOWN]
if shown:
    fig, axes = plt.subplots(2, len(shown), figsize=(2.2 * len(shown), 4.6), squeeze=False)
    for col, i in enumerate(shown):
        s = test_table.samples[i]
        image = load_mri_image(s.path)
        axes[0, col].imshow(image, cmap="gray")
        axes[0, col].set_title(f"{y_test[i]}\n{image.shape[1]}x{image.shape[0]}", fontsize=7)
        axes[1, col].imshow(prepare_image(image, config.preprocessing).cnn_image, cmap="gray", vmin=0, vmax=1)
        axes[1, col].set_title(f"conf {confidence[i]:.2f}", fontsize=7)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()